# 06 — Refit

**Purpose:** Retrain each winning model on the full dataset (train + test combined).  
**Input:** `*_train.csv` + `*_test.csv` for each model  
**Output:** Overwritten `.pkl` files in `models/` — these are the production models  

**Rules:**
- Exact same hyperparameters as the winners selected in 04a/04b/04c — no tuning
- No evaluation here — evaluation is strictly 05_combined_eval's job

| Model | Winner | Train target |
|-------|--------|-------------|
| offered | Gradient Boosting | `offered` |
| capacity | Random Forest | `log_capacity` |
| enrollment | Random Forest | `log_enrolled` |

In [1]:
import joblib
import numpy as np
import pandas as pd
import sqlite3
from pathlib import Path
from sklearn.ensemble import GradientBoostingClassifier, RandomForestRegressor

DATA_PATH   = Path('../data/processed')
MODELS_PATH = Path('../models')
MODELS_PATH.mkdir(exist_ok=True)

# feature lists — must match 04a/04b/04c exactly
FEATURES_OFFERED = [
    'ml_course_id', 'dept_code_enc', 'degree_level_enc', 'course_level', 'units',
    'term_order', 'is_covid_affected',
    'hist_n_offerings', 'hist_n_this_semester_offerings', 'same_semester_offer_ratio',
    'n_distinct_semesters_offered', 'n_terms_since_last_offered',
    'n_consecutive_same_semester_streak',
]
FEATURES_CAPACITY = [
    'ml_course_id', 'dept_code_enc', 'degree_level_enc', 'course_level', 'units',
    'term_order', 'is_covid_affected',
    'hist_avg_capacity_per_offering', 'hist_avg_capacity_this_semester',
    'same_semester_capacity_ratio', 'previous_term_capacity',
    'previous_same_semester_capacity', 'capacity_trend',
    'hist_n_offerings', 'hist_avg_sections_per_offering',
    'hist_avg_enrollment_per_offering',
]
FEATURES_ENROLLMENT = [
    'ml_course_id', 'dept_code_enc', 'degree_level_enc', 'course_level', 'units',
    'term_order', 'is_covid_affected',
    'hist_avg_capacity_per_offering', 'hist_avg_capacity_this_semester',
    'hist_avg_enrollment_per_offering', 'hist_avg_enrollment_this_semester',
    'same_semester_enrollment_ratio', 'previous_same_semester_enrollment',
    'previous_term_enrollment', 'enrollment_trend',
    'hist_n_offerings', 'hist_avg_sections_per_offering',
    'high_fill_rate_frequency', 'prereq_count', 'course_age_terms',
]

print('ready')

ready


---
## Block 1 — Load and combine train + test

In [2]:
# offered
offered_train = pd.read_csv(DATA_PATH / 'offered_train.csv')
offered_test  = pd.read_csv(DATA_PATH / 'offered_test.csv')
offered_full  = pd.concat([offered_train, offered_test], ignore_index=True)

# capacity
capacity_train = pd.read_csv(DATA_PATH / 'capacity_train.csv')
capacity_test  = pd.read_csv(DATA_PATH / 'capacity_test.csv')
capacity_full  = pd.concat([capacity_train, capacity_test], ignore_index=True)

# enrollment
enrollment_train = pd.read_csv(DATA_PATH / 'enrollment_train.csv')
enrollment_test  = pd.read_csv(DATA_PATH / 'enrollment_test.csv')
enrollment_full  = pd.concat([enrollment_train, enrollment_test], ignore_index=True)

print('=== COMBINED DATASET SIZES ===')
print(f'offered:    {len(offered_full):>6,} rows  (train {len(offered_train):,} + test {len(offered_test):,})')
print(f'capacity:   {len(capacity_full):>6,} rows  (train {len(capacity_train):,} + test {len(capacity_test):,})')
print(f'enrollment: {len(enrollment_full):>6,} rows  (train {len(enrollment_train):,} + test {len(enrollment_test):,})')

=== COMBINED DATASET SIZES ===
offered:    48,907 rows  (train 45,840 + test 3,067)
capacity:   24,376 rows  (train 22,854 + test 1,522)
enrollment: 24,376 rows  (train 22,854 + test 1,522)


---
## Block 2 — Refit offered model (Gradient Boosting)

In [3]:
X_offered = offered_full[FEATURES_OFFERED]
y_offered = offered_full['offered']

# exact same hyperparameters as 04a winner
# pos_weight recomputed from combined data (more data = better estimate of true class balance)
pos_weight     = (y_offered == 0).sum() / (y_offered == 1).sum()
sample_weights = np.where(y_offered == 1, pos_weight, 1.0)

model_offered = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    random_state=42
)
model_offered.fit(X_offered, y_offered, sample_weight=sample_weights)

joblib.dump(model_offered, MODELS_PATH / 'model_offered.pkl')
print(f'offered model refit on {len(X_offered):,} rows')
print(f'positive rate: {y_offered.mean()*100:.1f}%')
print(f'pos_weight used: {pos_weight:.4f}')
print(f'saved: model_offered.pkl')

offered model refit on 48,907 rows
positive rate: 49.8%
pos_weight used: 1.0064
saved: model_offered.pkl


---
## Block 3 — Refit capacity model (Random Forest)

In [4]:
X_capacity = capacity_full[FEATURES_CAPACITY]
y_capacity = capacity_full['log_capacity']

# exact same hyperparameters as 04b winner
model_capacity = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
model_capacity.fit(X_capacity, y_capacity)

joblib.dump(model_capacity, MODELS_PATH / 'model_capacity.pkl')
print(f'capacity model refit on {len(X_capacity):,} rows')
print(f'mean log_capacity: {y_capacity.mean():.3f}  (real: {np.expm1(y_capacity.mean()):.1f} seats)')
print(f'saved: model_capacity.pkl')

capacity model refit on 24,376 rows
mean log_capacity: 3.646  (real: 37.3 seats)
saved: model_capacity.pkl


---
## Block 4 — Refit enrollment model (Random Forest)

In [5]:
X_enrollment = enrollment_full[FEATURES_ENROLLMENT]
y_enrollment = enrollment_full['log_enrolled']

# exact same hyperparameters as 04c winner
model_enrollment = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
model_enrollment.fit(X_enrollment, y_enrollment)

joblib.dump(model_enrollment, MODELS_PATH / 'model_enrollment.pkl')
print(f'enrollment model refit on {len(X_enrollment):,} rows')
print(f'mean log_enrolled: {y_enrollment.mean():.3f}  (real: {np.expm1(y_enrollment.mean()):.1f} students)')
print(f'saved: model_enrollment.pkl')

enrollment model refit on 24,376 rows
mean log_enrolled: 3.146  (real: 22.2 students)
saved: model_enrollment.pkl


---
## Block 5 — Sanity check

Not a formal evaluation — just confirming models load and produce reasonable outputs.  
Pick CMPT 225 from the test set and check predictions make sense.

In [6]:
# find CMPT 225 ml_course_id
conn = sqlite3.connect('../data/processed/sfu_clean.db')
cmpt225_id = pd.read_sql(
    "SELECT DISTINCT ml_course_id FROM offerings WHERE dept_code='CMPT' AND course_number='225' LIMIT 1",
    conn
).iloc[0, 0]
conn.close()
print(f'CMPT 225 ml_course_id: {cmpt225_id}')

# look it up in test set (the term the refit model has never been evaluated on)
cmpt225_offered = offered_test[offered_test['ml_course_id'] == cmpt225_id]
cmpt225_cap     = capacity_test[capacity_test['ml_course_id'] == cmpt225_id]
cmpt225_enr     = enrollment_test[enrollment_test['ml_course_id'] == cmpt225_id]

print(f'CMPT 225 rows in test set — offered: {len(cmpt225_offered)}, capacity: {len(cmpt225_cap)}, enrollment: {len(cmpt225_enr)}')

CMPT 225 ml_course_id: 523
CMPT 225 rows in test set — offered: 1, capacity: 1, enrollment: 1


In [7]:
if len(cmpt225_offered) > 0:
    row_off      = cmpt225_offered[FEATURES_OFFERED].iloc[[0]]
    offered_prob = model_offered.predict_proba(row_off)[0, 1]
    offered_pred = model_offered.predict(row_off)[0]
    actual_off   = cmpt225_offered['offered'].iloc[0]

    print('CMPT 225 — test term sanity check:')
    print(f'  Offered prob:        {offered_prob:.3f}  → predicted: {"offered" if offered_pred == 1 else "not offered"}')
    print(f'  Actual:              {"offered" if actual_off == 1 else "not offered"}')

    if len(cmpt225_cap) > 0:
        cap_pred   = np.expm1(model_capacity.predict(cmpt225_cap[FEATURES_CAPACITY].iloc[[0]]))[0]
        cap_actual = cmpt225_cap['total_capacity'].iloc[0]
        print(f'  Predicted capacity:  {cap_pred:.0f} seats     Actual: {cap_actual:.0f} seats')

    if len(cmpt225_enr) > 0:
        enr_pred   = np.expm1(model_enrollment.predict(cmpt225_enr[FEATURES_ENROLLMENT].iloc[[0]]))[0]
        enr_actual = cmpt225_enr['total_enrolled'].iloc[0]
        print(f'  Predicted enrollment:{enr_pred:.0f} students  Actual: {enr_actual:.0f} students')
else:
    print('CMPT 225 not found in test set')

CMPT 225 — test term sanity check:
  Offered prob:        0.950  → predicted: offered
  Actual:              offered
  Predicted capacity:  403 seats     Actual: 400 seats
  Predicted enrollment:282 students  Actual: 265 students


In [8]:
# confirm all three model files exist and sizes look right
print('=== SAVED MODEL FILES ===')
for name in ['model_offered.pkl', 'model_capacity.pkl', 'model_enrollment.pkl']:
    path = MODELS_PATH / name
    if path.exists():
        size_kb = path.stat().st_size // 1024
        print(f'  {name:<30} {size_kb:>6} KB')
    else:
        print(f'  {name:<30} MISSING')

=== SAVED MODEL FILES ===
  model_offered.pkl                 345 KB
  model_capacity.pkl             154337 KB
  model_enrollment.pkl           185935 KB
